# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### 1. My lane as an ML task (type)

This is a **classification task** because the output is a binary label: whether a flight search resulted in a booking (yes) or did not result in a booking (no). The model’s job is to assign each search to one of these two categories.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target I would predict is **whether a flight search resulted in a booking**. This label comes from an **observed outcome** in the dataset: if the user completed a booking, the target is `1`; if not, the target is `0`.  

If direct booking data is missing, a reasonable **proxy** could be whether the user clicked on a flight offer, since clicks are a measurable signal of interest that often precede bookings.


In [2]:
import pandas as pd

# Load your lane's slice of the starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Inspect the first few rows
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

In [4]:
import os
os.getcwd()


'c:\\Users\\mokga\\Projects\\ml-internship-leonard\\work\\notebooks'

In [5]:
import pandas as pd

df = pd.read_csv(r"C:\Users\mokga\Projects\ml-internship-leonard\data\raw\content_refresh_anonymized.csv")
df.head()


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Because this is classification, a good success metric is **ROC‑AUC** (to measure how well the model separates engaged vs non‑engaged cases).  
Secondary metrics: **Precision/Recall** (to avoid false positives) and **F1 score** (to balance both).


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one **content refresh event**. Each row represents a unique `content_id` × `client_id` interaction with features describing the content and its performance.

In [6]:
import pandas as pd

# Load the dataset
df = pd.read_csv(r"C:\Users\mokga\Projects\ml-internship-leonard\data\raw\content_refresh_anonymized.csv")

# Inspect the first few rows
df.head()

# Sketch a target column (proxy for engagement)
df['engaged'] = df['engagement_rate'].apply(lambda x: 1 if x > 0 else 0)
df[['content_id', 'client_id', 'engagement_rate', 'engaged']].head()


,content_id,client_id,engagement_rate,engaged
0,content_304f48230142,client_f369cb89fc,5.88,1
1,content_a1fb4e703a9e,client_4e07408562,0.00,0
2,content_9aa793d4d895,client_7f2253d7e2,0.00,0
3,content_331d6c4de07b,client_19581e27de,1.28,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.00,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like `if engagement_rate > 0 then engaged = 1 else 0` is too simplistic.  
- Engagement depends on multiple interacting features: `ctr`, `avg_position`, `scroll_rate`, `competition_level`, `word_count`, etc.  
- These signals are noisy and sometimes contradictory. For example, a piece of content might have **low CTR but high scroll_rate**, or **high search_volume but low engagement_rate**.  
- An if‑statement can only capture one threshold at a time, but ML can learn **non‑linear patterns across 44 features** simultaneously.  
- ML models can generalize: they can recognize that certain combinations (e.g., high `search_volume` + medium `competition` + strong `trend_direction`) predict engagement even if no single feature crosses a threshold.

That’s why ML is better suited: it can discover messy, multi‑dimensional patterns that rules cannot.


### 6. The ML loop

The ML loop is the cycle of **data → model → evaluation → deployment → feedback**.  
For this content refresh dataset, the loop looks like:

1. **Data**  
   - Collect anonymized refresh events (`content_id`, `client_id`, features like `search_volume`, `ctr`, `avg_position`, `scroll_rate`).  
   - Define the target (`engaged` from `engagement_rate` or proxy signals).

2. **Model**  
   - Train a classifier (e.g., logistic regression, random forest, gradient boosting).  
   - Input = 44 features, Output = probability of engagement.

3. **Evaluation**  
   - Measure ROC‑AUC, Precision, Recall, F1.  
   - Compare against baseline rules (e.g., “if CTR > 0.5 then engaged”).

4. **Deployment**  
   - Integrate predictions into FlyRank’s ranking pipeline.  
   - Show higher‑probability content earlier.

5. **Feedback**  
   - Monitor real user behavior (clicks, scrolls, bookings).  
   - Feed new data back into training for continuous improvement.

This loop ensures the model adapts to changing trends, competition levels, and user intents — something fixed rules cannot do.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.